In [1]:
# Consolidated source exported from task1_nlp_analysis_clean.ipynb

# %% [Cell 1]
# ============================================================
# IU DLBDSEDA02 – Task 1 Development Phase
# NLP Topic Modelling of Consumer Complaint Narratives
#
# Kaggle dataset:
# https://www.kaggle.com/datasets/selener/consumer-complaint-database
#
# This notebook:
# - finds the attached Consumer Complaint Database automatically;
# - creates a reproducible sample of 10,000 unique narratives;
# - preprocesses the text with NLTK-compatible methods;
# - creates Bag-of-Words and TF-IDF matrices;
# - identifies an appropriate topic count through repeated
#   subsampling and term-centric stability analysis;
# - trains final LDA and NMF models;
# - creates tables, charts, README.md and requirements.txt;
# - saves all results under /kaggle/working/results.
# ============================================================


# ============================================================
# 1. IMPORTS AND SETTINGS
# ============================================================

import gc
import json
import math
import random
import re
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import nltk
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import RegexpTokenizer

from scipy.optimize import linear_sum_assignment

# Compatibility adjustment for some Kaggle SciPy/Gensim combinations.
try:
    import scipy.linalg as scipy_linalg

    if not hasattr(scipy_linalg, "triu"):
        scipy_linalg.triu = np.triu
except Exception:
    pass

from sklearn.decomposition import LatentDirichletAllocation, NMF
from sklearn.feature_extraction.text import (
    CountVectorizer,
    ENGLISH_STOP_WORDS,
    TfidfVectorizer,
)

try:
    from gensim.corpora import Dictionary
    from gensim.models import CoherenceModel
    GENSIM_AVAILABLE = True
except Exception as gensim_error:
    GENSIM_AVAILABLE = False
    print(
        "Gensim coherence will be skipped because Gensim could not be imported:",
        gensim_error,
    )


warnings.filterwarnings("ignore")

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

analysis_start_time = time.time()

SAMPLE_SIZE = 10_000
MAX_FEATURES = 5_000
MIN_DOCUMENT_FREQUENCY = 5
MAX_DOCUMENT_FREQUENCY = 0.95

MIN_TOPICS = 2
MAX_TOPICS = 20
TOPIC_VALUES = list(range(MIN_TOPICS, MAX_TOPICS + 1))

SUBSAMPLE_FRACTION = 0.80
STABILITY_RUNS = 5
TOP_WORDS = 10

LDA_STABILITY_ITERATIONS = 3
NMF_STABILITY_ITERATIONS = 120

FINAL_LDA_ITERATIONS = 15
FINAL_NMF_ITERATIONS = 400

COMPARABLE_SCORE_TOLERANCE = 0.01
MINIMUM_TOPIC_DIVERSITY = 0.70

INPUT_DIRECTORY = Path("/kaggle/input")
WORKING_DIRECTORY = Path("/kaggle/working")
RESULTS_DIRECTORY = WORKING_DIRECTORY / "results"

RESULTS_DIRECTORY.mkdir(parents=True, exist_ok=True)

GITHUB_REPOSITORY_URL = (
    "https://github.com/Leo-Steiner/"
    "Project-Task-1-iu-nlp-consumer-complaints-"
)

KAGGLE_NOTEBOOK_URL = (
    "https://www.kaggle.com/code/leosteiner0/notebookf59b1fdbbe"
)

KAGGLE_DATASET_URL = (
    "https://www.kaggle.com/datasets/selener/"
    "consumer-complaint-database"
)

NARRATIVE_COLUMN = "Consumer complaint narrative"
PRODUCT_COLUMN = "Product"

print("Python version:", sys.version.split()[0])
print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)
print("Results directory:", RESULTS_DIRECTORY)

# %% [Cell 2]
# ============================================================
# 2. HELPER FUNCTIONS
# ============================================================

def resource_available(resource_paths):
    """Return True if any supplied NLTK resource path is available."""
    for resource_path in resource_paths:
        try:
            nltk.data.find(resource_path)
            return True
        except LookupError:
            continue

    return False


STOPWORDS_AVAILABLE = resource_available(
    [
        "corpora/stopwords",
        "corpora/stopwords.zip",
    ]
)

WORDNET_AVAILABLE = resource_available(
    [
        "corpora/wordnet",
        "corpora/wordnet.zip",
    ]
)

if STOPWORDS_AVAILABLE:
    try:
        from nltk.corpus import stopwords

        STOP_WORDS = set(stopwords.words("english"))
        STOPWORD_SOURCE = "NLTK English stopwords"
    except Exception:
        STOP_WORDS = set(ENGLISH_STOP_WORDS)
        STOPWORD_SOURCE = "scikit-learn English stopwords fallback"
else:
    STOP_WORDS = set(ENGLISH_STOP_WORDS)
    STOPWORD_SOURCE = "scikit-learn English stopwords fallback"

DOMAIN_STOP_WORDS = {
    "xxxx",
    "xx",
    "xxx",
    "xxxxxxxx",
    "complaint",
    "consumer",
    "company",
    "please",
    "would",
    "could",
    "also",
    "said",
    "told",
    "received",
    "called",
    "contacted",
}

STOP_WORDS.update(DOMAIN_STOP_WORDS)

TOKENIZER = RegexpTokenizer(r"[a-z]+")
LEMMATIZER = WordNetLemmatizer()


def fallback_lemmatize(token):
    """
    Conservative rule-based fallback used only when the WordNet
    corpus is unavailable. It prevents Kaggle internet/resource
    errors while retaining reproducible normalization.
    """
    if len(token) <= 3:
        return token

    if token.endswith("ies") and len(token) > 4:
        return token[:-3] + "y"

    if token.endswith("sses"):
        return token[:-2]

    if token.endswith("xes") or token.endswith("zes"):
        return token[:-2]

    if token.endswith("ing") and len(token) > 5:
        stem = token[:-3]

        if (
            len(stem) > 2
            and stem[-1] == stem[-2]
            and stem[-1] not in {"s", "l"}
        ):
            stem = stem[:-1]

        return stem

    if token.endswith("ed") and len(token) > 4:
        stem = token[:-2]

        if (
            len(stem) > 2
            and stem[-1] == stem[-2]
            and stem[-1] not in {"s", "l"}
        ):
            stem = stem[:-1]

        return stem

    if token.endswith("s") and not token.endswith("ss"):
        return token[:-1]

    return token


def lemmatize_token(token):
    """Lemmatize safely without requiring a network download."""
    if WORDNET_AVAILABLE:
        try:
            return LEMMATIZER.lemmatize(
                LEMMATIZER.lemmatize(token, pos="v"),
                pos="n",
            )
        except LookupError:
            return fallback_lemmatize(token)

    return fallback_lemmatize(token)


def preprocess_text(text):
    """
    Lowercase text; remove URLs, redaction markers, numbers and
    punctuation; tokenize; remove stopwords; and lemmatize.
    """
    text = str(text).lower()

    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"\b[x]{2,}\b", " ", text)
    text = re.sub(r"\S+@\S+", " ", text)
    text = re.sub(r"\d+", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    tokens = TOKENIZER.tokenize(text)

    cleaned_tokens = []

    for token in tokens:
        if len(token) < 3:
            continue

        if token in STOP_WORDS:
            continue

        lemma = lemmatize_token(token)

        if len(lemma) < 3:
            continue

        if lemma in STOP_WORDS:
            continue

        cleaned_tokens.append(lemma)

    return " ".join(cleaned_tokens)


def get_top_words(model, feature_names, number_of_words=TOP_WORDS):
    """Extract ordered top-word lists from an LDA or NMF model."""
    topic_word_lists = []

    for component in model.components_:
        top_indices = component.argsort()[-number_of_words:][::-1]
        topic_word_lists.append(
            [str(feature_names[index]) for index in top_indices]
        )

    return topic_word_lists


def average_jaccard_similarity(ranked_words_a, ranked_words_b):
    """
    Calculate Average Jaccard ranked-list similarity.

    Agreement is evaluated at every prefix length and then averaged,
    so highly ranked shared words contribute throughout the score.
    """
    if not ranked_words_a or not ranked_words_b:
        return 0.0

    depth = min(len(ranked_words_a), len(ranked_words_b))

    prefix_scores = []

    for prefix_length in range(1, depth + 1):
        set_a = set(ranked_words_a[:prefix_length])
        set_b = set(ranked_words_b[:prefix_length])

        union = set_a | set_b

        if not union:
            prefix_scores.append(0.0)
        else:
            prefix_scores.append(len(set_a & set_b) / len(union))

    return float(np.mean(prefix_scores))


def matched_topic_similarity(reference_topics, candidate_topics):
    """
    Match topics one-to-one with the Hungarian algorithm and return
    mean ranked top-word agreement.
    """
    number_of_topics = len(reference_topics)

    similarity_matrix = np.zeros(
        (number_of_topics, number_of_topics),
        dtype=float,
    )

    for reference_index, reference_topic in enumerate(reference_topics):
        for candidate_index, candidate_topic in enumerate(candidate_topics):
            similarity_matrix[reference_index, candidate_index] = (
                average_jaccard_similarity(
                    reference_topic,
                    candidate_topic,
                )
            )

    reference_rows, candidate_columns = linear_sum_assignment(
        -similarity_matrix
    )

    matched_scores = similarity_matrix[
        reference_rows,
        candidate_columns,
    ]

    return float(matched_scores.mean()), similarity_matrix


def topic_diversity(topic_word_lists):
    """
    Calculate the proportion of unique terms among all displayed
    topic terms. Higher values indicate less duplication.
    """
    flattened_words = [
        word
        for topic_words in topic_word_lists
        for word in topic_words
    ]

    if not flattened_words:
        return 0.0

    return len(set(flattened_words)) / len(flattened_words)


def make_lda(
    number_of_topics,
    max_iterations,
    random_state=RANDOM_STATE,
):
    """Create an LDA model with Kaggle-safe settings."""
    return LatentDirichletAllocation(
        n_components=number_of_topics,
        learning_method="batch",
        max_iter=max_iterations,
        random_state=random_state,
        n_jobs=-1,
        evaluate_every=-1,
    )


def make_nmf(
    number_of_topics,
    max_iterations,
    random_state=RANDOM_STATE,
):
    """Create an NMF model with stable initialization."""
    return NMF(
        n_components=number_of_topics,
        init="nndsvda",
        solver="cd",
        beta_loss="frobenius",
        max_iter=max_iterations,
        random_state=random_state,
        alpha_W=0.0,
        alpha_H=0.0,
        l1_ratio=0.0,
    )


def safe_perplexity(model, matrix):
    """Return LDA perplexity or NaN if it cannot be calculated."""
    try:
        return float(model.perplexity(matrix))
    except Exception:
        return float("nan")


def calculate_coherence(
    tokenized_documents,
    topic_word_lists,
):
    """
    Calculate Gensim C_v coherence as a supporting interpretation
    check. It is not used to select the number of topics.
    """
    if not GENSIM_AVAILABLE:
        return float("nan")

    try:
        dictionary = Dictionary(tokenized_documents)

        coherence_model = CoherenceModel(
            topics=topic_word_lists,
            texts=tokenized_documents,
            dictionary=dictionary,
            coherence="c_v",
            processes=1,
        )

        return float(coherence_model.get_coherence())

    except Exception as error:
        print("Coherence calculation was skipped:", error)
        return float("nan")


def create_topic_labels(topic_word_lists, label_words=3):
    """Create transparent automatic topic labels from top terms."""
    labels = []

    for topic_index, words in enumerate(topic_word_lists, start=1):
        short_label = " / ".join(words[:label_words])
        labels.append(f"Topic {topic_index}: {short_label}")

    return labels


def save_dataframe(dataframe, filename):
    """Save a DataFrame under the results directory."""
    output_path = RESULTS_DIRECTORY / filename
    dataframe.to_csv(output_path, index=False)
    return output_path


def save_figure(figure_or_filename, filename=None):
    """
    Save a Matplotlib figure.

    Supports both:
    save_figure(filename)
    save_figure(figure, filename)
    """

    if filename is None:
        figure = plt.gcf()
        filename = figure_or_filename
    else:
        figure = figure_or_filename

    output_path = RESULTS_DIRECTORY / filename

    figure.tight_layout()

    figure.savefig(
        output_path,
        dpi=200,
        bbox_inches="tight",
    )

    plt.show()
    plt.close(figure)

    return output_path


def plot_topic_words(
    model,
    feature_names,
    model_name,
    filename,
    top_words=TOP_WORDS,
):
    """Create horizontal bar plots of the top words per topic."""
    number_of_topics = model.components_.shape[0]

    columns = 2
    rows = math.ceil(number_of_topics / columns)

    figure, axes = plt.subplots(
        rows,
        columns,
        figsize=(14, max(4, rows * 3.5)),
    )

    axes = np.atleast_1d(axes).flatten()

    for topic_index, component in enumerate(model.components_):
        top_indices = component.argsort()[-top_words:]
        words = feature_names[top_indices]
        weights = component[top_indices]

        axis = axes[topic_index]

        axis.barh(
            range(top_words),
            weights,
            color="#3569a8",
        )

        axis.set_yticks(range(top_words))
        axis.set_yticklabels(words)
        axis.set_title(
            f"{model_name} Topic {topic_index + 1}"
        )
        axis.set_xlabel("Term weight")

    for unused_axis in axes[number_of_topics:]:
        unused_axis.axis("off")

    figure.suptitle(
        f"{model_name}: Top {top_words} Words per Topic",
        fontsize=16,
        y=1.01,
    )

    figure.tight_layout()

    return save_figure(figure, filename)


def plot_topic_prevalence(
    prevalence_dataframe,
    title,
    filename,
    color,
):
    """Plot average document-topic prevalence."""
    figure, axis = plt.subplots(figsize=(11, 6))

    axis.bar(
        prevalence_dataframe["topic_label"],
        prevalence_dataframe["prevalence_percent"],
        color=color,
    )

    axis.set_title(title)
    axis.set_ylabel("Average topic prevalence (%)")
    axis.set_xlabel("Topic")
    axis.tick_params(axis="x", rotation=45)

    for label in axis.get_xticklabels():
        label.set_horizontalalignment("right")

    figure.tight_layout()

    return save_figure(figure, filename)

# %% [Cell 3]
# ============================================================
# 3. FIND THE DATASET AND LOAD A REPRODUCIBLE SAMPLE
# ============================================================

print("\nSearching for the Consumer Complaint Database...")

csv_files = sorted(INPUT_DIRECTORY.rglob("*.csv"))

if not csv_files:
    raise FileNotFoundError(
        "No CSV file was found. Use Add Input in Kaggle and attach "
        "the Consumer Complaint Database."
    )

dataset_path = None

for csv_file in csv_files:
    try:
        available_columns = pd.read_csv(
            csv_file,
            nrows=0,
            encoding="utf-8",
            encoding_errors="replace",
        ).columns.tolist()

        if NARRATIVE_COLUMN in available_columns:
            dataset_path = csv_file
            break

    except Exception:
        continue

if dataset_path is None:
    raise ValueError(
        f"None of the attached CSV files contains the required column "
        f"'{NARRATIVE_COLUMN}'. Attach the correct Consumer Complaint "
        f"Database from Kaggle."
    )

print(f"Correct dataset found: {dataset_path}")

available_columns = pd.read_csv(
    dataset_path,
    nrows=0,
    encoding="utf-8",
    encoding_errors="replace",
).columns.tolist()

columns_to_load = [NARRATIVE_COLUMN]

if PRODUCT_COLUMN in available_columns:
    columns_to_load.append(PRODUCT_COLUMN)

random_generator = np.random.default_rng(RANDOM_STATE)

retained_sample = pd.DataFrame()
seen_narrative_hashes = set()

source_rows = 0
missing_narratives = 0
duplicate_narratives = 0
valid_unique_narratives = 0

print("Reading the dataset in chunks...")

data_reader = pd.read_csv(
    dataset_path,
    usecols=columns_to_load,
    chunksize=50_000,
    encoding="utf-8",
    encoding_errors="replace",
    low_memory=False,
)

for chunk_number, chunk in enumerate(data_reader, start=1):
    source_rows += len(chunk)

    # Convert missing values to empty strings and remove surrounding spaces.
    chunk[NARRATIVE_COLUMN] = (
        chunk[NARRATIVE_COLUMN]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    # Remove missing or empty narratives before sampling.
    valid_narrative_mask = chunk[NARRATIVE_COLUMN].ne("")

    missing_narratives += int(
        (~valid_narrative_mask).sum()
    )

    chunk = chunk.loc[valid_narrative_mask].copy()

    if chunk.empty:
        continue

    # Ensure the Product column always exists for later sections.
    if PRODUCT_COLUMN not in chunk.columns:
        chunk[PRODUCT_COLUMN] = "Unknown"

    chunk[PRODUCT_COLUMN] = (
        chunk[PRODUCT_COLUMN]
        .fillna("Unknown")
        .astype(str)
    )

    # Remove duplicate narratives from the complete dataset.
    narrative_hashes = pd.util.hash_pandas_object(
        chunk[NARRATIVE_COLUMN],
        index=False,
    ).astype("uint64")

    unique_row_mask = []

    for narrative_hash in narrative_hashes:
        hash_value = int(narrative_hash)

        if hash_value in seen_narrative_hashes:
            unique_row_mask.append(False)
            duplicate_narratives += 1
        else:
            seen_narrative_hashes.add(hash_value)
            unique_row_mask.append(True)

    chunk = chunk.loc[unique_row_mask].copy()

    valid_unique_narratives += len(chunk)

    if chunk.empty:
        continue

    # Give every valid narrative a reproducible random priority.
    chunk["_sampling_priority"] = random_generator.random(
        len(chunk)
    )

    retained_sample = pd.concat(
        [retained_sample, chunk],
        ignore_index=True,
    )

    # Retain the 10,000 smallest random priorities.
    if len(retained_sample) > SAMPLE_SIZE:
        retained_sample = retained_sample.nsmallest(
            SAMPLE_SIZE,
            "_sampling_priority",
        ).copy()

    if chunk_number % 10 == 0:
        print(
            f"Processed {source_rows:,} rows; "
            f"found {valid_unique_narratives:,} "
            f"valid unique narratives."
        )

if retained_sample.empty:
    raise ValueError(
        "No usable complaint narratives were found. "
        "Check whether the correct dataset was attached."
    )

complaints = (
    retained_sample
    .nsmallest(
        min(SAMPLE_SIZE, len(retained_sample)),
        "_sampling_priority",
    )
    .drop(columns="_sampling_priority")
    .reset_index(drop=True)
)

raw_sample_size = int(len(complaints))

loading_summary = {
    "source_rows": int(source_rows),

    # Names used by the later notebook sections
    "missing_narratives": int(missing_narratives),
    "duplicate_narratives": int(duplicate_narratives),

    # Additional descriptive names
    "missing_narratives_removed": int(missing_narratives),
    "duplicate_narratives_removed": int(duplicate_narratives),

    "unique_valid_narratives": int(valid_unique_narratives),
    "sample_size": int(len(complaints)),
    "random_state": RANDOM_STATE,
}

print("\nDataset loading completed.")
print(f"Source rows examined: {source_rows:,}")
print(f"Missing narratives removed: {missing_narratives:,}")
print(f"Duplicate narratives removed: {duplicate_narratives:,}")
print(f"Valid unique narratives: {valid_unique_narratives:,}")
print(f"Final sample size: {len(complaints):,}")

if len(complaints) < 100:
    raise ValueError(
        "The attached dataset genuinely contains fewer than 100 valid "
        "complaint narratives. Confirm that you attached this dataset: "
        "https://www.kaggle.com/datasets/selener/"
        "consumer-complaint-database"
    )

# %% [Cell 4]
# ============================================================
# 4. PREPROCESS THE COMPLAINT NARRATIVES
# ============================================================

print("\nPreprocessing complaint narratives...")
print("Stopword source:", STOPWORD_SOURCE)

if WORDNET_AVAILABLE:
    print("Lemmatization source: NLTK WordNetLemmatizer")
else:
    print(
        "Lemmatization source: safe rule-based fallback "
        "(WordNet corpus was not available)"
    )

if len(complaints) < 100:
    raise ValueError(
        "The dataset-loading section supplied fewer than 100 narratives. "
        "Confirm that the correct Consumer Complaint Database is attached."
    )

preprocessing_start = time.time()

complaints["raw_word_count"] = (
    complaints[NARRATIVE_COLUMN]
    .fillna("")
    .astype(str)
    .str.split()
    .str.len()
)

complaints["clean_text"] = complaints[NARRATIVE_COLUMN].map(preprocess_text)

complaints["clean_word_count"] = (
    complaints["clean_text"].str.split().str.len()
)

empty_after_cleaning = int(
    complaints["clean_text"].str.strip().eq("").sum()
)

complaints = complaints.loc[
    complaints["clean_text"].str.strip().ne("")
].reset_index(drop=True)

if len(complaints) < 100:
    raise ValueError(
        "Fewer than 100 narratives remained after preprocessing. "
        "Review the attached dataset and preprocessing output."
    )

preprocessing_seconds = time.time() - preprocessing_start

example_columns = [NARRATIVE_COLUMN, "clean_text"]
if PRODUCT_COLUMN in complaints.columns:
    example_columns.append(PRODUCT_COLUMN)

after_examples = complaints[example_columns].head(3).copy()

print(f"Preprocessing completed in {preprocessing_seconds / 60:.2f} minutes.")
print(f"Usable narratives after preprocessing: {len(complaints):,}")
print("\nThree before-and-after cleaning examples:")

for example_number, row in after_examples.iterrows():
    print(f"\nExample {example_number + 1}")
    print("Before:")
    print(str(row[NARRATIVE_COLUMN])[:500])
    print("\nAfter:")
    print(str(row["clean_text"])[:500])

# %% [Cell 5]
# ============================================================
# 5. DATA-QUALITY TABLES
# ============================================================

cleaning_summary = pd.DataFrame(
    {
        "measure": [
            "Rows in source CSV",
            "Missing narratives removed",
            "Duplicate narratives removed",
            "Requested sample size",
            "Sampled narratives",
            "Narratives empty after preprocessing",
            "Narratives used for modelling",
            "Random state",
        ],
        "value": [
            loading_summary["source_rows"],
            loading_summary["missing_narratives"],
            loading_summary["duplicate_narratives"],
            SAMPLE_SIZE,
            raw_sample_size,
            empty_after_cleaning,
            len(complaints),
            RANDOM_STATE,
        ],
    }
)

save_dataframe(
    cleaning_summary,
    "cleaning_summary.csv",
)

data_quality_summary = pd.DataFrame(
    {
        "measure": [
            "Mean raw words per narrative",
            "Median raw words per narrative",
            "Mean cleaned words per narrative",
            "Median cleaned words per narrative",
            "Minimum cleaned words",
            "Maximum cleaned words",
        ],
        "value": [
            round(float(complaints["raw_word_count"].mean()), 2),
            round(float(complaints["raw_word_count"].median()), 2),
            round(float(complaints["clean_word_count"].mean()), 2),
            round(float(complaints["clean_word_count"].median()), 2),
            int(complaints["clean_word_count"].min()),
            int(complaints["clean_word_count"].max()),
        ],
    }
)

save_dataframe(
    data_quality_summary,
    "data_quality_summary.csv",
)

# %% [Cell 6]
# ============================================================
# 6. BAG-OF-WORDS AND TF-IDF VECTORIZATION
# ============================================================

print("\nCreating Bag-of-Words and TF-IDF matrices...")

effective_min_df = MIN_DOCUMENT_FREQUENCY

if len(complaints) < 1_000:
    effective_min_df = 2

count_vectorizer = CountVectorizer(
    max_features=MAX_FEATURES,
    min_df=effective_min_df,
    max_df=MAX_DOCUMENT_FREQUENCY,
)

tfidf_vectorizer = TfidfVectorizer(
    max_features=MAX_FEATURES,
    min_df=effective_min_df,
    max_df=MAX_DOCUMENT_FREQUENCY,
    sublinear_tf=True,
    norm="l2",
)

count_matrix = count_vectorizer.fit_transform(
    complaints["clean_text"]
)

tfidf_matrix = tfidf_vectorizer.fit_transform(
    complaints["clean_text"]
)

if count_matrix.shape[1] < MAX_TOPICS:
    raise ValueError(
        "The CountVectorizer vocabulary is smaller than the maximum "
        "topic count. Check the dataset and preprocessing."
    )

if tfidf_matrix.shape[1] < MAX_TOPICS:
    raise ValueError(
        "The TF-IDF vocabulary is smaller than the maximum topic "
        "count. Check the dataset and preprocessing."
    )

count_feature_names = (
    count_vectorizer.get_feature_names_out()
)

tfidf_feature_names = (
    tfidf_vectorizer.get_feature_names_out()
)

print("Bag-of-Words shape:", count_matrix.shape)
print("TF-IDF shape:", tfidf_matrix.shape)

count_term_frequencies = np.asarray(
    count_matrix.sum(axis=0)
).ravel()

top_count_indices = count_term_frequencies.argsort()[-20:][::-1]

top_count_terms = pd.DataFrame(
    {
        "term": count_feature_names[top_count_indices],
        "count": count_term_frequencies[top_count_indices],
    }
)

save_dataframe(
    top_count_terms,
    "top_count_terms.csv",
)

tfidf_term_scores = np.asarray(
    tfidf_matrix.mean(axis=0)
).ravel()

top_tfidf_indices = tfidf_term_scores.argsort()[-20:][::-1]

top_tfidf_terms = pd.DataFrame(
    {
        "term": tfidf_feature_names[top_tfidf_indices],
        "mean_tfidf": tfidf_term_scores[top_tfidf_indices],
    }
)

save_dataframe(
    top_tfidf_terms,
    "top_tfidf_terms.csv",
)

vectorization_comparison = pd.DataFrame(
    {
        "method": [
            "Bag of Words",
            "TF-IDF",
        ],
        "documents": [
            count_matrix.shape[0],
            tfidf_matrix.shape[0],
        ],
        "features": [
            count_matrix.shape[1],
            tfidf_matrix.shape[1],
        ],
        "nonzero_values": [
            count_matrix.nnz,
            tfidf_matrix.nnz,
        ],
        "density_percent": [
            round(
                100
                * count_matrix.nnz
                / (
                    count_matrix.shape[0]
                    * count_matrix.shape[1]
                ),
                4,
            ),
            round(
                100
                * tfidf_matrix.nnz
                / (
                    tfidf_matrix.shape[0]
                    * tfidf_matrix.shape[1]
                ),
                4,
            ),
        ],
    }
)

save_dataframe(
    vectorization_comparison,
    "vectorization_comparison.csv",
)

# %% [Cell 7]
# ============================================================
# 7. TERM-CENTRIC TOPIC-STABILITY ANALYSIS
# ============================================================

print(
    "\nIdentifying the appropriate number of topics through "
    "repeated 80% subsampling and ranked top-word stability."
)

print(
    f"Topic range: {MIN_TOPICS}–{MAX_TOPICS}; "
    f"repeated subsamples per topic count: {STABILITY_RUNS}."
)

document_count = count_matrix.shape[0]

subsample_size = max(
    100,
    int(round(document_count * SUBSAMPLE_FRACTION)),
)

subsample_size = min(
    subsample_size,
    document_count,
)

stability_records = []

stability_start_time = time.time()

for number_of_topics in TOPIC_VALUES:
    topic_start_time = time.time()

    print(
        f"\nEvaluating {number_of_topics} topics "
        f"({number_of_topics - MIN_TOPICS + 1}/"
        f"{len(TOPIC_VALUES)})..."
    )

    reference_lda = make_lda(
        number_of_topics=number_of_topics,
        max_iterations=LDA_STABILITY_ITERATIONS,
        random_state=RANDOM_STATE,
    )

    reference_lda.fit(count_matrix)

    reference_lda_topics = get_top_words(
        reference_lda,
        count_feature_names,
        TOP_WORDS,
    )

    reference_nmf = make_nmf(
        number_of_topics=number_of_topics,
        max_iterations=NMF_STABILITY_ITERATIONS,
        random_state=RANDOM_STATE,
    )

    reference_nmf.fit(tfidf_matrix)

    reference_nmf_topics = get_top_words(
        reference_nmf,
        tfidf_feature_names,
        TOP_WORDS,
    )

    lda_stability_scores = []
    nmf_stability_scores = []

    for stability_run in range(STABILITY_RUNS):
        run_seed = (
            RANDOM_STATE
            + number_of_topics * 1_000
            + stability_run
        )

        run_rng = np.random.default_rng(run_seed)

        subsample_indices = np.sort(
            run_rng.choice(
                document_count,
                size=subsample_size,
                replace=False,
            )
        )

        lda_subsample_model = make_lda(
            number_of_topics=number_of_topics,
            max_iterations=LDA_STABILITY_ITERATIONS,
            random_state=run_seed,
        )

        lda_subsample_model.fit(
            count_matrix[subsample_indices]
        )

        lda_subsample_topics = get_top_words(
            lda_subsample_model,
            count_feature_names,
            TOP_WORDS,
        )

        lda_agreement, _ = matched_topic_similarity(
            reference_lda_topics,
            lda_subsample_topics,
        )

        lda_stability_scores.append(lda_agreement)

        nmf_subsample_model = make_nmf(
            number_of_topics=number_of_topics,
            max_iterations=NMF_STABILITY_ITERATIONS,
            random_state=run_seed,
        )

        nmf_subsample_model.fit(
            tfidf_matrix[subsample_indices]
        )

        nmf_subsample_topics = get_top_words(
            nmf_subsample_model,
            tfidf_feature_names,
            TOP_WORDS,
        )

        nmf_agreement, _ = matched_topic_similarity(
            reference_nmf_topics,
            nmf_subsample_topics,
        )

        nmf_stability_scores.append(nmf_agreement)

        print(
            f"  Run {stability_run + 1}/{STABILITY_RUNS}: "
            f"LDA={lda_agreement:.3f}, "
            f"NMF={nmf_agreement:.3f}"
        )

        del (
            lda_subsample_model,
            nmf_subsample_model,
            lda_subsample_topics,
            nmf_subsample_topics,
        )

        gc.collect()

    lda_stability_mean = float(
        np.mean(lda_stability_scores)
    )

    nmf_stability_mean = float(
        np.mean(nmf_stability_scores)
    )

    lda_stability_std = float(
        np.std(lda_stability_scores, ddof=1)
        if len(lda_stability_scores) > 1
        else 0.0
    )

    nmf_stability_std = float(
        np.std(nmf_stability_scores, ddof=1)
        if len(nmf_stability_scores) > 1
        else 0.0
    )

    combined_run_scores = (
        np.asarray(lda_stability_scores)
        + np.asarray(nmf_stability_scores)
    ) / 2

    combined_stability = float(
        combined_run_scores.mean()
    )

    combined_stability_std = float(
        combined_run_scores.std(ddof=1)
        if len(combined_run_scores) > 1
        else 0.0
    )

    combined_standard_error = (
        combined_stability_std
        / math.sqrt(STABILITY_RUNS)
    )

    lda_diversity = topic_diversity(
        reference_lda_topics
    )

    nmf_diversity = topic_diversity(
        reference_nmf_topics
    )

    combined_diversity = float(
        (lda_diversity + nmf_diversity) / 2
    )

    lda_perplexity = safe_perplexity(
        reference_lda,
        count_matrix,
    )

    nmf_reconstruction_error = float(
        reference_nmf.reconstruction_err_
    )

    topic_elapsed_seconds = (
        time.time() - topic_start_time
    )

    stability_records.append(
        {
            "number_of_topics": number_of_topics,
            "lda_stability_mean": lda_stability_mean,
            "lda_stability_std": lda_stability_std,
            "nmf_stability_mean": nmf_stability_mean,
            "nmf_stability_std": nmf_stability_std,
            "combined_stability": combined_stability,
            "combined_stability_std": (
                combined_stability_std
            ),
            "combined_standard_error": (
                combined_standard_error
            ),
            "lda_topic_diversity": lda_diversity,
            "nmf_topic_diversity": nmf_diversity,
            "combined_topic_diversity": (
                combined_diversity
            ),
            "lda_perplexity_supporting_metric": (
                lda_perplexity
            ),
            "nmf_reconstruction_error_supporting_metric": (
                nmf_reconstruction_error
            ),
            "evaluation_seconds": (
                topic_elapsed_seconds
            ),
        }
    )

    print(
        f"  Mean stability: "
        f"LDA={lda_stability_mean:.3f}, "
        f"NMF={nmf_stability_mean:.3f}, "
        f"combined={combined_stability:.3f}"
    )

    print(
        f"  Combined topic diversity: "
        f"{combined_diversity:.3f}"
    )

    print(
        f"  Completed in "
        f"{topic_elapsed_seconds / 60:.2f} minutes."
    )

    del reference_lda
    del reference_nmf
    gc.collect()

model_evaluation = pd.DataFrame(
    stability_records
).sort_values("number_of_topics").reset_index(drop=True)

maximum_combined_stability = float(
    model_evaluation["combined_stability"].max()
)

best_score_row = model_evaluation.loc[
    model_evaluation["combined_stability"].idxmax()
]

selection_tolerance = max(
    COMPARABLE_SCORE_TOLERANCE,
    float(best_score_row["combined_standard_error"]),
)

comparable_threshold = (
    maximum_combined_stability - selection_tolerance
)

model_evaluation["comparable_to_best"] = (
    model_evaluation["combined_stability"]
    >= comparable_threshold
)

model_evaluation["passes_diversity_check"] = (
    model_evaluation["combined_topic_diversity"]
    >= MINIMUM_TOPIC_DIVERSITY
)

eligible_topic_solutions = model_evaluation.loc[
    model_evaluation["comparable_to_best"]
    & model_evaluation["passes_diversity_check"]
].copy()

if eligible_topic_solutions.empty:
    eligible_topic_solutions = model_evaluation.loc[
        model_evaluation["comparable_to_best"]
    ].copy()

if eligible_topic_solutions.empty:
    eligible_topic_solutions = model_evaluation.loc[
        model_evaluation["combined_stability"]
        == maximum_combined_stability
    ].copy()

selected_topic_count = int(
    eligible_topic_solutions[
        "number_of_topics"
    ].min()
)

model_evaluation["selected_solution"] = (
    model_evaluation["number_of_topics"]
    == selected_topic_count
)

save_dataframe(
    model_evaluation,
    "model_evaluation.csv",
)

stability_elapsed_minutes = (
    time.time() - stability_start_time
) / 60

print(
    f"\nStability analysis completed in "
    f"{stability_elapsed_minutes:.2f} minutes."
)

print(
    f"Maximum combined stability: "
    f"{maximum_combined_stability:.4f}"
)

print(
    f"Comparable-score tolerance: "
    f"{selection_tolerance:.4f}"
)

print(
    f"Comparable threshold: "
    f"{comparable_threshold:.4f}"
)

print(
    f"Selected topic count: "
    f"{selected_topic_count}"
)

print(
    "Selection rule: choose the smallest sufficiently diverse "
    "solution whose combined stability is comparable to the "
    "highest observed stability."
)

# %% [Cell 8]
# ============================================================
# 8. PLOT TOPIC-COUNT EVALUATION
# ============================================================

figure, axis = plt.subplots(figsize=(12, 7))

axis.plot(
    model_evaluation["number_of_topics"],
    model_evaluation["lda_stability_mean"],
    marker="o",
    linewidth=2,
    label="LDA stability",
    color="#3569a8",
)

axis.plot(
    model_evaluation["number_of_topics"],
    model_evaluation["nmf_stability_mean"],
    marker="s",
    linewidth=2,
    label="NMF stability",
    color="#d78326",
)

axis.plot(
    model_evaluation["number_of_topics"],
    model_evaluation["combined_stability"],
    marker="D",
    linewidth=3,
    label="Combined stability",
    color="#2c8c5a",
)

axis.fill_between(
    model_evaluation["number_of_topics"],
    (
        model_evaluation["combined_stability"]
        - model_evaluation["combined_standard_error"]
    ),
    (
        model_evaluation["combined_stability"]
        + model_evaluation["combined_standard_error"]
    ),
    color="#2c8c5a",
    alpha=0.15,
    label="Combined ± 1 standard error",
)

axis.axhline(
    comparable_threshold,
    color="#777777",
    linestyle="--",
    linewidth=1.5,
    label="Comparable-score threshold",
)

axis.axvline(
    selected_topic_count,
    color="#b22222",
    linestyle="--",
    linewidth=2,
    label=f"Selected k = {selected_topic_count}",
)

selected_row = model_evaluation.loc[
    model_evaluation["number_of_topics"]
    == selected_topic_count
].iloc[0]

axis.scatter(
    [selected_topic_count],
    [selected_row["combined_stability"]],
    s=150,
    color="#b22222",
    zorder=5,
)

axis.set_title(
    "Topic-Count Selection by Repeated Subsample Stability"
)

axis.set_xlabel("Number of topics (k)")
axis.set_ylabel("Ranked top-word stability")
axis.set_xticks(TOPIC_VALUES)
axis.set_ylim(
    0,
    min(
        1.0,
        max(
            0.5,
            float(
                model_evaluation[
                    "combined_stability"
                ].max()
            )
            + 0.12,
        ),
    ),
)

axis.grid(alpha=0.25)
axis.legend(loc="best")

figure.tight_layout()

save_figure(
    figure,
    "model_evaluation.png",
)

# %% [Cell 9]
# ============================================================
# 9. TRAIN FINAL LDA AND NMF MODELS
# ============================================================

print(
    f"\nTraining final LDA and NMF models with "
    f"{selected_topic_count} topics..."
)

final_lda = make_lda(
    number_of_topics=selected_topic_count,
    max_iterations=FINAL_LDA_ITERATIONS,
    random_state=RANDOM_STATE,
)

lda_document_topics = final_lda.fit_transform(
    count_matrix
)

final_nmf = make_nmf(
    number_of_topics=selected_topic_count,
    max_iterations=FINAL_NMF_ITERATIONS,
    random_state=RANDOM_STATE,
)

nmf_document_topics = final_nmf.fit_transform(
    tfidf_matrix
)

final_lda_topics = get_top_words(
    final_lda,
    count_feature_names,
    TOP_WORDS,
)

final_nmf_topics = get_top_words(
    final_nmf,
    tfidf_feature_names,
    TOP_WORDS,
)

lda_labels = create_topic_labels(
    final_lda_topics,
    label_words=3,
)

nmf_labels = create_topic_labels(
    final_nmf_topics,
    label_words=3,
)

print("\nFinal LDA topics:")

for topic_number, words in enumerate(
    final_lda_topics,
    start=1,
):
    print(
        f"LDA Topic {topic_number}: "
        f"{', '.join(words)}"
    )

print("\nFinal NMF topics:")

for topic_number, words in enumerate(
    final_nmf_topics,
    start=1,
):
    print(
        f"NMF Topic {topic_number}: "
        f"{', '.join(words)}"
    )

# %% [Cell 10]
# ============================================================
# 10. SUPPORTING COHERENCE CHECK
# ============================================================

tokenized_documents = [
    text.split()
    for text in complaints["clean_text"]
]

lda_coherence = calculate_coherence(
    tokenized_documents,
    final_lda_topics,
)

nmf_coherence = calculate_coherence(
    tokenized_documents,
    final_nmf_topics,
)

print("\nSupporting C_v coherence results:")

if np.isfinite(lda_coherence):
    print(f"LDA C_v coherence: {lda_coherence:.4f}")
else:
    print("LDA C_v coherence: not available")

if np.isfinite(nmf_coherence):
    print(f"NMF C_v coherence: {nmf_coherence:.4f}")
else:
    print("NMF C_v coherence: not available")

# %% [Cell 11]
# ============================================================
# 11. CREATE TOPIC-WORD TABLE
# ============================================================

topic_word_rows = []

for model_name, topic_lists in [
    ("LDA", final_lda_topics),
    ("NMF", final_nmf_topics),
]:
    for topic_index, words in enumerate(
        topic_lists,
        start=1,
    ):
        for rank, word in enumerate(words, start=1):
            topic_word_rows.append(
                {
                    "model": model_name,
                    "topic_number": topic_index,
                    "rank": rank,
                    "word": word,
                }
            )

topic_words_dataframe = pd.DataFrame(
    topic_word_rows
)

save_dataframe(
    topic_words_dataframe,
    "topic_words.csv",
)

# %% [Cell 12]
# ============================================================
# 12. CALCULATE TOPIC PREVALENCE
# ============================================================

lda_prevalence = lda_document_topics.mean(axis=0)

nmf_row_sums = nmf_document_topics.sum(axis=1, keepdims=True)
nmf_document_topic_shares = np.divide(
    nmf_document_topics,
    nmf_row_sums,
    out=np.zeros_like(nmf_document_topics),
    where=nmf_row_sums != 0,
)
nmf_prevalence = nmf_document_topic_shares.mean(axis=0)

if nmf_prevalence.sum() > 0:
    nmf_prevalence = nmf_prevalence / nmf_prevalence.sum()

lda_prevalence_dataframe = pd.DataFrame(
    {
        "topic_number": np.arange(
            1,
            selected_topic_count + 1,
        ),
        "topic_label": lda_labels,
        "prevalence": lda_prevalence,
        "prevalence_percent": (
            lda_prevalence * 100
        ),
        "top_words": [
            ", ".join(words)
            for words in final_lda_topics
        ],
    }
).sort_values(
    "prevalence",
    ascending=False,
).reset_index(drop=True)

nmf_prevalence_dataframe = pd.DataFrame(
    {
        "topic_number": np.arange(
            1,
            selected_topic_count + 1,
        ),
        "topic_label": nmf_labels,
        "prevalence": nmf_prevalence,
        "prevalence_percent": (
            nmf_prevalence * 100
        ),
        "top_words": [
            ", ".join(words)
            for words in final_nmf_topics
        ],
    }
).sort_values(
    "prevalence",
    ascending=False,
).reset_index(drop=True)

save_dataframe(
    lda_prevalence_dataframe,
    "lda_topic_prevalence.csv",
)

save_dataframe(
    nmf_prevalence_dataframe,
    "nmf_topic_prevalence.csv",
)

# %% [Cell 13]
# ============================================================
# 13. COMPARE FINAL LDA AND NMF TOPICS
# ============================================================

final_model_agreement, final_similarity_matrix = (
    matched_topic_similarity(
        final_lda_topics,
        final_nmf_topics,
    )
)

lda_topic_indices, nmf_topic_indices = (
    linear_sum_assignment(-final_similarity_matrix)
)

comparison_rows = []

for lda_index, nmf_index in zip(
    lda_topic_indices,
    nmf_topic_indices,
):
    comparison_rows.append(
        {
            "lda_topic_number": int(lda_index + 1),
            "nmf_topic_number": int(nmf_index + 1),
            "ranked_top_word_agreement": float(
                final_similarity_matrix[
                    lda_index,
                    nmf_index,
                ]
            ),
            "lda_top_words": ", ".join(
                final_lda_topics[lda_index]
            ),
            "nmf_top_words": ", ".join(
                final_nmf_topics[nmf_index]
            ),
            "shared_words": ", ".join(
                sorted(
                    set(final_lda_topics[lda_index])
                    & set(final_nmf_topics[nmf_index])
                )
            ),
        }
    )

lda_nmf_topic_comparison = pd.DataFrame(
    comparison_rows
).sort_values(
    "lda_topic_number"
).reset_index(drop=True)

save_dataframe(
    lda_nmf_topic_comparison,
    "lda_nmf_topic_comparison.csv",
)

print(
    f"\nMean matched LDA–NMF top-word agreement: "
    f"{final_model_agreement:.4f}"
)

# %% [Cell 14]
# ============================================================
# 14. CREATE FINAL CHARTS
# ============================================================

plot_topic_words(
    model=final_lda,
    feature_names=count_feature_names,
    model_name="LDA",
    filename="lda_topic_words.png",
)

plot_topic_words(
    model=final_nmf,
    feature_names=tfidf_feature_names,
    model_name="NMF",
    filename="nmf_topic_words.png",
)

plot_topic_prevalence(
    prevalence_dataframe=lda_prevalence_dataframe,
    title="LDA Topic Prevalence",
    filename="lda_topic_prevalence.png",
    color="#3569a8",
)

plot_topic_prevalence(
    prevalence_dataframe=nmf_prevalence_dataframe,
    title="NMF Topic Prevalence",
    filename="nmf_topic_prevalence.png",
    color="#d78326",
)

figure, axis = plt.subplots(
    figsize=(10, 8)
)

heatmap = axis.imshow(
    final_similarity_matrix,
    cmap="Blues",
    vmin=0,
    vmax=1,
    aspect="auto",
)

axis.set_title(
    "Ranked Top-Word Agreement Between Final LDA and NMF Topics"
)

axis.set_xlabel("NMF topic")
axis.set_ylabel("LDA topic")

axis.set_xticks(
    np.arange(selected_topic_count)
)

axis.set_yticks(
    np.arange(selected_topic_count)
)

axis.set_xticklabels(
    np.arange(1, selected_topic_count + 1)
)

axis.set_yticklabels(
    np.arange(1, selected_topic_count + 1)
)

for row_index in range(selected_topic_count):
    for column_index in range(selected_topic_count):
        score = final_similarity_matrix[
            row_index,
            column_index,
        ]

        text_color = (
            "white"
            if score >= 0.50
            else "black"
        )

        axis.text(
            column_index,
            row_index,
            f"{score:.2f}",
            ha="center",
            va="center",
            color=text_color,
            fontsize=9,
        )

figure.colorbar(
    heatmap,
    ax=axis,
    label="Average Jaccard ranked-list similarity",
)

figure.tight_layout()

save_figure(
    figure,
    "lda_nmf_topic_overlap.png",
)


if PRODUCT_COLUMN in complaints.columns:
    product_counts = (
        complaints[PRODUCT_COLUMN]
        .fillna("Not available")
        .value_counts()
        .head(10)
        .sort_values()
    )

    figure, axis = plt.subplots(figsize=(12, 7))
    product_counts.plot.barh(ax=axis, color="#5B9BD5")
    axis.set_xlabel("Number of sampled complaints")
    axis.set_ylabel("Product")
    axis.set_title("Ten Most Frequent Products in the Analysis Sample")
    figure.tight_layout()
    save_figure(figure, "top_products.png")

# %% [Cell 15]
# ============================================================
# 15. SAVE A MACHINE-READABLE RESULT SUMMARY
# ============================================================

selected_evaluation_row = model_evaluation.loc[
    model_evaluation["number_of_topics"] == selected_topic_count
].iloc[0]


def finite_or_none(value):
    """Return a JSON-safe float or None."""
    numeric_value = float(value)
    return numeric_value if np.isfinite(numeric_value) else None


analysis_runtime_minutes = (time.time() - analysis_start_time) / 60

result_summary = {
    "project": "NLP Analysis of Consumer Complaints",
    "student": "Leo Steiner",
    "matriculation_number": "14130885",
    "random_state": RANDOM_STATE,
    "dataset_path": str(dataset_path),
    "source_rows": loading_summary["source_rows"],
    "sample_size_before_preprocessing": raw_sample_size,
    "documents_after_preprocessing": int(len(complaints)),
    "count_features": int(count_matrix.shape[1]),
    "tfidf_features": int(tfidf_matrix.shape[1]),
    "candidate_topic_numbers": TOPIC_VALUES,
    "stability_runs": STABILITY_RUNS,
    "subsample_fraction": SUBSAMPLE_FRACTION,
    "selection_method": (
        "Term-centric stability across repeated 80% subsamples, "
        "Hungarian topic matching and Average Jaccard ranked-word agreement"
    ),
    "selected_number_of_topics": selected_topic_count,
    "maximum_combined_stability": maximum_combined_stability,
    "comparable_score_tolerance": selection_tolerance,
    "comparable_threshold": comparable_threshold,
    "selected_lda_stability": float(
        selected_evaluation_row["lda_stability_mean"]
    ),
    "selected_nmf_stability": float(
        selected_evaluation_row["nmf_stability_mean"]
    ),
    "selected_combined_stability": float(
        selected_evaluation_row["combined_stability"]
    ),
    "selected_topic_diversity": float(
        selected_evaluation_row["combined_topic_diversity"]
    ),
    "final_lda_nmf_similarity": final_model_agreement,
    "lda_cv_coherence": finite_or_none(lda_coherence),
    "nmf_cv_coherence": finite_or_none(nmf_coherence),
    "analysis_runtime_minutes": analysis_runtime_minutes,
    "lda_topics": lda_prevalence_dataframe.to_dict(orient="records"),
    "nmf_topics": nmf_prevalence_dataframe.to_dict(orient="records"),
}

result_summary_path = RESULTS_DIRECTORY / "result_summary.json"

with open(result_summary_path, "w", encoding="utf-8") as result_file:
    json.dump(result_summary, result_file, indent=2, ensure_ascii=False)

print(f"Result summary saved to: {result_summary_path}")

# %% [Cell 16]
# ============================================================
# 16. CREATE requirements.txt WITH THE EXECUTED VERSIONS
# ============================================================

from importlib.metadata import PackageNotFoundError, version

required_packages = [
    "numpy",
    "pandas",
    "nltk",
    "scipy",
    "scikit-learn",
    "matplotlib",
    "gensim",
]

requirement_lines = []

for package_name in required_packages:
    try:
        requirement_lines.append(f"{package_name}=={version(package_name)}")
    except PackageNotFoundError:
        # Gensim is optional at runtime; the core stability analysis does not
        # depend on it. Listing the package without a version still documents
        # the intended dependency if a Kaggle image omits it.
        requirement_lines.append(package_name)

requirements_path = WORKING_DIRECTORY / "requirements.txt"
requirements_path.write_text(
    "\n".join(requirement_lines) + "\n",
    encoding="utf-8",
)

print("requirements.txt created:")
print(requirements_path.read_text(encoding="utf-8"))

# %% [Cell 17]
# ============================================================
# 17. CREATE THE COMPLETE GITHUB README
# ============================================================


def create_readme_topic_lines(dataframe, model_name):
    """Create accurate topic lines from this run's prevalence table."""
    return "\n".join(
        (
            f"- **{model_name} topic {int(row.topic_number)} "
            f"({row.topic_label}):** {row.top_words} "
            f"({float(row.prevalence_percent):.2f}% average prevalence)"
        )
        for row in dataframe.itertuples(index=False)
    )


lda_topic_lines = create_readme_topic_lines(
    lda_prevalence_dataframe,
    "LDA",
)

nmf_topic_lines = create_readme_topic_lines(
    nmf_prevalence_dataframe,
    "NMF",
)

lda_coherence_text = (
    f"{lda_coherence:.4f}" if np.isfinite(lda_coherence) else "not available"
)
nmf_coherence_text = (
    f"{nmf_coherence:.4f}" if np.isfinite(nmf_coherence) else "not available"
)

readme_content = f"""# NLP Analysis of Consumer Complaints

## Project overview

This repository contains the development phase of IU course DLBDSEDA02
Task 1. The project uses natural language processing and unsupervised topic
modelling to identify prevalent themes in consumer complaint narratives.

## Dataset

The analysis uses the Consumer Complaint Database available through Kaggle.
The original dataset is not stored in this repository because of its size.

- Dataset: {KAGGLE_DATASET_URL}
- Online notebook: {KAGGLE_NOTEBOOK_URL}

A reproducible sample of {raw_sample_size:,} unique, non-empty narratives was
created with random state {RANDOM_STATE}. After preprocessing,
{len(complaints):,} documents remained.

## Data preparation

The workflow removes missing and duplicate narratives, converts text to
lowercase, removes URLs, email addresses, numbers, punctuation and masked
`XXXX` content, tokenizes the text, removes English stopwords and lemmatizes
the remaining words.

## Vectorization

- CountVectorizer created a Bag-of-Words matrix with
  {count_matrix.shape[1]:,} features for LDA.
- TF-IDF created a weighted matrix with {tfidf_matrix.shape[1]:,} features for
  NMF, reducing the influence of terms that occur in many documents.

The vectorization statistics and highest-ranked terms are stored in the
`results` folder.

## Selecting the number of topics

The topic count was identified through term-centric stability analysis rather
than through subjective inspection. For every candidate from {MIN_TOPICS} to
{MAX_TOPICS}, LDA and NMF were fitted to {STABILITY_RUNS} repeated
{SUBSAMPLE_FRACTION:.0%} subsamples. Topics were matched with the Hungarian
algorithm, and ranked top-word agreement was measured with Average Jaccard
similarity.

The highest combined stability was {maximum_combined_stability:.4f}. The
defined comparability and diversity rule selected **{selected_topic_count}
topics**, with combined stability
{float(selected_evaluation_row['combined_stability']):.4f}.

## Main results

The mean matched LDA-NMF top-word agreement was
{final_model_agreement:.4f}. Supporting C_v coherence was
{lda_coherence_text} for LDA and {nmf_coherence_text} for NMF.

### LDA topics

{lda_topic_lines}

### NMF topics

{nmf_topic_lines}

## Repository structure

~~~text
README.md
requirements.txt
task1_nlp_analysis.ipynb
results/
    cleaning_summary.csv
    data_quality_summary.csv
    vectorization_comparison.csv
    top_count_terms.csv
    top_tfidf_terms.csv
    model_evaluation.csv
    model_evaluation.png
    topic_words.csv
    lda_topic_prevalence.csv
    lda_topic_prevalence.png
    lda_topic_words.png
    nmf_topic_prevalence.csv
    nmf_topic_prevalence.png
    nmf_topic_words.png
    lda_nmf_topic_comparison.csv
    lda_nmf_topic_overlap.png
    top_products.png
    result_summary.json
~~~

## Run the analysis

1. Open the Kaggle notebook.
2. Attach the Consumer Complaint Database through **Add Input**.
3. Use a standard CPU session.
4. Select **Run All**.
5. Download the completed notebook and `/kaggle/working` outputs.
6. Upload the notebook, README, requirements and results to GitHub.

The notebook is designed to run without internet access. If an optional NLTK
corpus or Gensim is unavailable, a documented fallback is used without
changing the core stability-based selection method.

## Limitations

- Sampling can exclude rare complaint themes.
- Stability measures reproducibility, not guaranteed semantic usefulness.
- Preprocessing choices influence the resulting topic structure.
- Automatic topic labels are transparent summaries of the three
  highest-weighted terms and may require human interpretation.
- The selected topic count applies to this sample and modelling configuration.

## Project links

- GitHub repository: {GITHUB_REPOSITORY_URL}
- Kaggle notebook: {KAGGLE_NOTEBOOK_URL}
- Kaggle dataset: {KAGGLE_DATASET_URL}

## References

Blei, D. M., Ng, A. Y., and Jordan, M. I. (2003). Latent Dirichlet
allocation. *Journal of Machine Learning Research, 3*, 993-1022.

Greene, D., O'Callaghan, D., and Cunningham, P. (2014). How many topics?
Stability analysis for topic models. In *Machine Learning and Knowledge
Discovery in Databases* (pp. 498-513). Springer.

Lee, D. D., and Seung, H. S. (1999). Learning the parts of objects by
non-negative matrix factorization. *Nature, 401*, 788-791.
"""

readme_path = WORKING_DIRECTORY / "README.md"
readme_path.write_text(readme_content, encoding="utf-8")

if readme_path.stat().st_size == 0:
    raise RuntimeError("README.md was created but is empty.")

print(f"README.md created: {readme_path} ({readme_path.stat().st_size:,} bytes)")

# %% [Cell 18]
# ============================================================
# 18. VALIDATE THE COMPLETE DEVELOPMENT-PHASE OUTPUT
# ============================================================

required_output_files = [
    WORKING_DIRECTORY / "README.md",
    WORKING_DIRECTORY / "requirements.txt",
    RESULTS_DIRECTORY / "cleaning_summary.csv",
    RESULTS_DIRECTORY / "data_quality_summary.csv",
    RESULTS_DIRECTORY / "vectorization_comparison.csv",
    RESULTS_DIRECTORY / "top_count_terms.csv",
    RESULTS_DIRECTORY / "top_tfidf_terms.csv",
    RESULTS_DIRECTORY / "model_evaluation.csv",
    RESULTS_DIRECTORY / "model_evaluation.png",
    RESULTS_DIRECTORY / "topic_words.csv",
    RESULTS_DIRECTORY / "lda_topic_prevalence.csv",
    RESULTS_DIRECTORY / "lda_topic_prevalence.png",
    RESULTS_DIRECTORY / "lda_topic_words.png",
    RESULTS_DIRECTORY / "nmf_topic_prevalence.csv",
    RESULTS_DIRECTORY / "nmf_topic_prevalence.png",
    RESULTS_DIRECTORY / "nmf_topic_words.png",
    RESULTS_DIRECTORY / "lda_nmf_topic_comparison.csv",
    RESULTS_DIRECTORY / "lda_nmf_topic_overlap.png",
    RESULTS_DIRECTORY / "result_summary.json",
]

if PRODUCT_COLUMN in complaints.columns:
    required_output_files.append(RESULTS_DIRECTORY / "top_products.png")

missing_or_empty_files = [
    str(path)
    for path in required_output_files
    if not path.exists() or path.stat().st_size == 0
]

if missing_or_empty_files:
    raise RuntimeError(
        "The following required outputs are missing or empty:\n"
        + "\n".join(missing_or_empty_files)
    )

if model_evaluation["number_of_topics"].tolist() != TOPIC_VALUES:
    raise RuntimeError("The model evaluation does not contain every candidate k.")

if len(lda_prevalence_dataframe) != selected_topic_count:
    raise RuntimeError("The LDA prevalence table has an incorrect topic count.")

if len(nmf_prevalence_dataframe) != selected_topic_count:
    raise RuntimeError("The NMF prevalence table has an incorrect topic count.")

if not np.isclose(lda_prevalence_dataframe["prevalence"].sum(), 1.0):
    raise RuntimeError("LDA prevalence values do not sum to one.")

if not np.isclose(nmf_prevalence_dataframe["prevalence"].sum(), 1.0):
    raise RuntimeError("NMF prevalence values do not sum to one.")

print("\n" + "=" * 72)
print("DEVELOPMENT-PHASE ANALYSIS COMPLETED AND VALIDATED")
print("=" * 72)
print(f"Documents modelled: {len(complaints):,}")
print(f"Selected topic count: {selected_topic_count}")
print(
    "Selected combined stability: "
    f"{float(selected_evaluation_row['combined_stability']):.4f}"
)
print(f"Generated and checked {len(required_output_files)} output files.")

for output_path in required_output_files:
    print(
        f"- {output_path.relative_to(WORKING_DIRECTORY)} "
        f"({output_path.stat().st_size:,} bytes)"
    )


Python version: 3.12.13
Pandas version: 2.3.3
NumPy version: 2.0.2
Results directory: /kaggle/working/results

Searching for the Consumer Complaint Database...
Correct dataset found: /kaggle/input/datasets/selener/consumer-complaint-database/rows.csv
Reading the dataset in chunks...
Processed 500,000 rows; found 199,075 valid unique narratives.

Dataset loading completed.
Source rows examined: 1,282,355
Missing narratives removed: 898,791
Duplicate narratives removed: 16,619
Valid unique narratives: 366,945
Final sample size: 10,000

Preprocessing complaint narratives...
Stopword source: NLTK English stopwords
Lemmatization source: NLTK WordNetLemmatizer
Preprocessing completed in 0.21 minutes.
Usable narratives after preprocessing: 10,000

Three before-and-after cleaning examples:

Example 1
Before:
The complaint is with XXXX XXXX and Navient and has XXXX distinct partsServicer Issues:1. The servicers had discussed my loan issues/represented a negative connotations about my loan to fa